# Best Regression Models for Model Fitting

This notebook demonstrates three powerful regression models that provide excellent model fitting capabilities for predictive modeling tasks. We'll implement and compare these models using cross-validation and hyperparameter tuning to find the optimal model for your data.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 2. Load and Explore Data

Let's load the Toyota Corolla dataset and explore its structure.

In [ ]:
# Load the dataset
file_path = "D:\\Data science\\Assignments\\Questions\\MLR\\MLR\\ToyotaCorolla - MLR.csv"
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Check data types and missing values
df.info()

In [ ]:
# Summary statistics
df.describe()

## 3. Data Preprocessing

Let's prepare our data for modeling by handling categorical variables and scaling numerical features.

In [ ]:
# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target variable from features
if 'Price' in numerical_cols:
    numerical_cols.remove('Price')

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Split data
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## 4. Exploratory Data Analysis

Let's visualize the relationships between features and the target variable.

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(12, 10))
corr = df[numerical_cols + ['Price']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for key numerical features
key_features = ['Price', 'Age_08_04', 'KM', 'HP', 'Weight']
sns.pairplot(df[key_features])
plt.suptitle('Pairplot of Key Features', y=1.02)
plt.show()

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 6))
sns.histplot(df['Price'], kde=True)
plt.title('Distribution of Car Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

## 5. Model 1: Elastic Net Regression

Elastic Net combines L1 and L2 regularization to handle multicollinearity and perform feature selection. It's particularly useful when you have many correlated features.

In [ ]:
# Create pipeline with preprocessing and model
elastic_net_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', ElasticNet(random_state=42))
])

# Define parameter grid for GridSearchCV
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    elastic_net_pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Get best parameters
best_params = grid_search.best_params_
print("Elastic Net - Best Parameters:")
print(best_params)

In [ ]:
# Evaluate the model
y_pred_en = grid_search.predict(X_test)
mse_en = mean_squared_error(y_test, y_pred_en)
rmse_en = np.sqrt(mse_en)
mae_en = mean_absolute_error(y_test, y_pred_en)
r2_en = r2_score(y_test, y_pred_en)

print(f"Elastic Net Performance:")
print(f"MSE: {mse_en:.2f}")
print(f"RMSE: {rmse_en:.2f}")
print(f"MAE: {mae_en:.2f}")
print(f"R² Score: {r2_en:.4f}")

In [ ]:
# Feature importance (coefficients)
feature_names = preprocessor.get_feature_names_out()
coefficients = grid_search.best_estimator_.named_steps['model'].coef_

# Create a DataFrame for feature importance
feature_importance_en = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})
feature_importance_en = feature_importance_en.sort_values('Coefficient', key=abs, ascending=False)

# Plot top 10 features
plt.figure(figsize=(12, 8))
top_features = feature_importance_en.head(10)
sns.barplot(x='Coefficient', y='Feature', data=top_features)
plt.title('Top 10 Features - Elastic Net')
plt.tight_layout()
plt.show()

# Display top features
feature_importance_en.head(10)

## 6. Model 2: Gradient Boosting Regressor

Gradient Boosting builds an ensemble of decision trees sequentially, where each tree corrects the errors of the previous ones. It's excellent for capturing complex non-linear relationships.

In [ ]:
# Create pipeline with preprocessing and model
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

# Define parameter grid for GridSearchCV
param_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    gb_pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Get best parameters
best_params = grid_search.best_params_
print("Gradient Boosting - Best Parameters:")
print(best_params)

In [ ]:
# Evaluate the model
y_pred_gb = grid_search.predict(X_test)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f"Gradient Boosting Performance:")
print(f"MSE: {mse_gb:.2f}")
print(f"RMSE: {rmse_gb:.2f}")
print(f"MAE: {mae_gb:.2f}")
print(f"R² Score: {r2_gb:.4f}")

In [ ]:
# Feature importance
feature_names = preprocessor.get_feature_names_out()
feature_importance_gb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': grid_search.best_estimator_.named_steps['model'].feature_importances_
})
feature_importance_gb = feature_importance_gb.sort_values('Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(12, 8))
top_features = feature_importance_gb.head(10)
sns.barplot(x='Importance', y='Feature', data=top_features)
plt.title('Top 10 Features - Gradient Boosting')
plt.tight_layout()
plt.show()

# Display top features
feature_importance_gb.head(10)

## 7. Model 3: Random Forest Regressor

Random Forest builds an ensemble of decision trees in parallel and averages their predictions. It's robust to outliers and can capture complex interactions between features.

In [ ]:
# Create pipeline with preprocessing and model
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Define parameter grid for GridSearchCV
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    rf_pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Get best parameters
best_params = grid_search.best_params_
print("Random Forest - Best Parameters:")
print(best_params)

In [ ]:
# Evaluate the model
y_pred_rf = grid_search.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Performance:")
print(f"MSE: {mse_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"MAE: {mae_rf:.2f}")
print(f"R² Score: {r2_rf:.4f}")

In [ ]:
# Feature importance
feature_names = preprocessor.get_feature_names_out()
feature_importance_rf = pd.DataFrame({
    'Feature': feature_names,
    'Importance': grid_search.best_estimator_.named_steps['model'].feature_importances_
})
feature_importance_rf = feature_importance_rf.sort_values('Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(12, 8))
top_features = feature_importance_rf.head(10)
sns.barplot(x='Importance', y='Feature', data=top_features)
plt.title('Top 10 Features - Random Forest')
plt.tight_layout()
plt.show()

# Display top features
feature_importance_rf.head(10)

## 8. Model Comparison

Let's compare the performance of all three models to determine which one provides the best fit for our data.

In [ ]:
# Create a DataFrame for model comparison
model_comparison = pd.DataFrame({
    'Model': ['Elastic Net', 'Gradient Boosting', 'Random Forest'],
    'MSE': [mse_en, mse_gb, mse_rf],
    'RMSE': [rmse_en, rmse_gb, rmse_rf],
    'MAE': [mae_en, mae_gb, mae_rf],
    'R² Score': [r2_en, r2_gb, r2_rf]
})

# Sort by R² score
model_comparison = model_comparison.sort_values('R² Score', ascending=False)

model_comparison

In [ ]:
# Plot model comparison
plt.figure(figsize=(14, 10))

# R² Score comparison
plt.subplot(2, 2, 1)
sns.barplot(x='Model', y='R² Score', data=model_comparison)
plt.title('R² Score Comparison')
plt.ylim(0, 1)

# MSE comparison
plt.subplot(2, 2, 2)
sns.barplot(x='Model', y='MSE', data=model_comparison)
plt.title('MSE Comparison')

# RMSE comparison
plt.subplot(2, 2, 3)
sns.barplot(x='Model', y='RMSE', data=model_comparison)
plt.title('RMSE Comparison')

# MAE comparison
plt.subplot(2, 2, 4)
sns.barplot(x='Model', y='MAE', data=model_comparison)
plt.title('MAE Comparison')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions vs actual values for the best model
best_model = model_comparison.iloc[0]['Model']

if best_model == 'Elastic Net':
    y_pred = y_pred_en
elif best_model == 'Gradient Boosting':
    y_pred = y_pred_gb
else:  # Random Forest
    y_pred = y_pred_rf

plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted Prices - {best_model}')
plt.tight_layout()
plt.show()

## 9. Residual Analysis

Let's analyze the residuals of our best model to check for any patterns or issues.

In [ ]:
# Calculate residuals
residuals = y_test - y_pred

# Plot residuals
plt.figure(figsize=(14, 10))

# Residuals vs Predicted values
plt.subplot(2, 2, 1)
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values')

# Histogram of residuals
plt.subplot(2, 2, 2)
sns.histplot(residuals, kde=True)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals')

# Q-Q plot
plt.subplot(2, 2, 3)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')

# Residuals vs Actual values
plt.subplot(2, 2, 4)
plt.scatter(y_test, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Actual Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Actual Values')

plt.tight_layout()
plt.show()

## 10. Conclusion

Based on our analysis, we can draw the following conclusions:

1. **Best Model**: The model comparison shows that [best model name] provides the best fit for our data with the highest R² score and lowest error metrics.

2. **Key Features**: The most important features for predicting car prices are [list top features from best model].

3. **Model Performance**: Our best model achieves an R² score of [best R² score], indicating that it explains a significant portion of the variance in car prices.

4. **Residual Analysis**: The residual analysis shows [observations about residuals - e.g., normally distributed, no clear patterns].

These regression models demonstrate different approaches to fitting the data, each with its own strengths:

- **Elastic Net**: Combines L1 and L2 regularization to handle multicollinearity and perform feature selection.
- **Gradient Boosting**: Builds an ensemble of decision trees sequentially to capture complex non-linear relationships.
- **Random Forest**: Creates an ensemble of decision trees in parallel, robust to outliers and good at capturing feature interactions.

For future work, we could:

1. Try additional models like XGBoost or LightGBM
2. Perform more extensive hyperparameter tuning
3. Engineer additional features that might improve model performance
4. Collect more data to further improve model accuracy